In [ ]:
import pandas as pd
import numpy as np
from utils.data import *

GLOBAL_START='2024-01-01'
GLOBAL_END='2026-03-01'

In [6]:

price_chain_df, hsi_df, vhsi_df, future = get_data()
price_chain_df_clean, hsi_df_clean, vhsi_df_clean, index_df = prepare_datasets(
    price_chain_df, hsi_df, vhsi_df, future
)
merged_df=pd.merge(price_chain_df_clean, index_df, on="Date", how="left")
index_features=add_index_features(merged_df)
smirk_daily = fit_iv_surface_shape_daily(
    df=price_chain_df_clean,                  # 你的 option chain dataframe
    price_col="SettlementPrice",   # 用来算 parity 的价格列
    iv_col="ImpliedVolatility",
    rate_col=None,           # 没有无风险利率就先 None
    r_default=0.0,
    dte_col="dte",
    x_band=np.inf,
    min_points=5,
)

option_features=add_option_shape_daily_features(
    shape_df=smirk_daily)

agg_features = build_periodic_features(
    index_features=index_features,
    option_features=option_features,
    lookback=3,   # 1M history
    pred_len=1,   # predict next 1M regime
)



/Users/ostrichzhang/Desktop/HKUST/25-26Spring/6000C/MFIT6000CAlgoTrading/utils/data.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hsi[col] = hsi[col].fillna(method="ffill")


21.325049999999997 8.733954870122249 170.39373054131107 0.518675376403693
21.350749999999998 1.8977888611646874 91.62866424693236 0.6909998592699018
22.0883 -0.8917452089551113 43.066679119053966 0.6241099022972733
21.6032 0.6288768531050396 54.444638437111514 0.32783154765931755
21.886049999999997 -7.38098494241786 30.858136585944543 0.4874425751474846
22.0332 -5.988864393120134 16.444834031396233 0.29605043124075175
21.940649999999998 -7.642054308881613 1.7035202339497852 0.5001733848753975
21.83605 -6.015834172741403 -0.6957890542033408 0.0706026562647522
21.8039 -4.854380391852041 -0.350657165998846 0.3049582501650979
21.8098 -8.216685886042256 3.2618794792099957 0.049841157710714554
21.8103 -8.70816633650625 -0.9971880576127305 0.2232939254273523
20.97725 8.227642272388694 183.57111350272976 0.55234065067246
21.35325 2.0983279673759694 76.69161696920997 0.6817544976292513
21.9281 -2.061524752257199 46.90532253198793 0.7497492409325303
21.72025 -4.324912734817907 48.033462386827125

In [ ]:
from utils.context import build_news_context_from_db
news_context = await build_news_context_from_db(
    start_date="2024-08-01",
    end_date="2024-09-01",
    keyword=None,
    limit=100,
    fulltext_chars=300,
)

In [ ]:
from utils.llm import call_llm, choose_scenario

reasoning, res=call_llm(as_of_date="2024-09-01")
scenario=choose_scenario(res)


In [ ]:
from utils.backtest import compute_backtest_metrics

metrics=compute_backtest_metrics(scenario, entry_date="2024-08-01", exit_date="2024-09-01", option_df=merged_df)


In [ ]:
from db.operations import save_performance_row
save_performance_row(metrics)